# MedGemma-4B QLoRA 파인튜닝

## Google Colab 무료 T4 환경용

`google/medgemma-4b-it`를 QLoRA 방식으로 파인튜닝하는 Notebook입니다.

### 주요 구성

* MedGemma 4B Instruct
* 4bit 양자화
* LoRA / QLoRA
* Google Colab T4 GPU
* KorMedMCQA doctor 데이터셋
* Google Drive 체크포인트 저장
* Hugging Face Hub LoRA Adapter 업로드

### 실행 순서

반드시 아래 순서대로 실행하세요.

1. GPU 확인
2. 패키지 설치
3. 런타임 재시작
4. 환경 확인
5. Hugging Face 로그인
6. 데이터셋 로드
7. 데이터셋 포맷팅
8. MedGemma 로드
9. LoRA 설정
10. Google Drive 연결
11. 학습
12. 필요하면 체크포인트에서 재개
13. 추론 테스트
14. Hugging Face Hub 업로드

### 중요

패키지 설치 셀 실행 후에는 반드시:

`런타임 → 세션 다시 시작`

또는

`런타임 → 런타임 다시 시작`

을 실행하세요.

Colab에서 이미 로드된 NumPy와 새로 설치된 NumPy가 충돌하는 것을 방지하기 위한 과정입니다.


---

# 0. GPU 확인


In [ ]:
!nvidia-smi


정상적으로 T4 GPU가 연결되어 있다면 다음과 비슷한 정보가 출력됩니다.

```text
Tesla T4
```

GPU가 없다면 이후 MedGemma 학습을 진행하지 마세요.

---

# 1. 패키지 설치

## 중요 (numpy는 건드리지 않습니다)

한때 `numpy.dtype size changed, may indicate binary incompatibility` 에러 때문에
numpy를 특정 버전(1.26.4)으로 낮춰 고정했었는데, 이게 오히려 문제였습니다 — 지금 Colab
기본 이미지는 `opencv`, `jax`, `cupy`, `shap`, `cudf`, `tifffile`, `rasterio` 등
수십 개 사전 설치 패키지가 전부 `numpy>=2`를 요구하도록 이미 맞춰져 있어서, numpy를 2.0
밑으로 내리면 오히려 그 패키지들과 어긋나 같은 종류의 ABI 에러가 재발합니다.

그래서 **numpy 버전은 아예 지정하지 않고, Colab에 이미 깔린 걸 그대로 씁니다.** 우리가 필요한
패키지만 설치합니다.


In [ ]:
%pip install -q -U transformers accelerate peft bitsandbytes datasets huggingface_hub trl

# torchvision은 이 노트북(텍스트 전용 QLoRA 파인튜닝)에 필요 없음.
# Colab 기본 이미지의 torch/torchvision 버전이 서로 안 맞는 경우가 있어서
# (torch 2.13.0 vs torchvision이 요구하는 torch 2.11.0), 남겨두면 MedGemma처럼
# 멀티모달 모델 클래스를 로드할 때 torchvision::nms 관련 에러로 막힌다. 지워서 회피.
!pip uninstall -y -q torchvision


---

# 2. 반드시 런타임 재시작

## 이 셀은 설명용입니다.

패키지 설치가 완료되면 아래 메뉴를 직접 실행하세요.

```text
런타임
→ 런타임 다시 시작
```

또는 Colab UI에 따라:

```text
런타임
→ 세션 다시 시작
```

재시작 후 아래 셀부터 다시 실행합니다.

---

# 3. Python / NumPy / GPU 환경 확인


In [ ]:
import sys
import numpy
import torch

print("Python :", sys.version)
print("NumPy  :", numpy.__version__)
print("PyTorch:", torch.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)


정상적인 예 (numpy 버전은 Colab이 준 값 그대로면 됨, 특정 값을 강제하지 않음):

```text
CUDA available: True
GPU: Tesla T4
```

---

# 4. 핵심 패키지 Import 테스트

앞서 겪었던 numpy ABI 충돌이 없는지 먼저 확인합니다. 여기서 에러가 나면 numpy를 건드리는
다른 셀/명령을 실행한 적이 있는지 먼저 의심하고, `런타임 다시 시작` 후 재시도하세요.


In [ ]:
import numpy
import pandas
import pyarrow
import scipy

print("NumPy   :", numpy.__version__)
print("Pandas  :", pandas.__version__)
print("PyArrow :", pyarrow.__version__)
print("SciPy   :", scipy.__version__)

from datasets import load_dataset

print("datasets import OK")


여기서:

```text
datasets import OK
```

가 출력되어야 합니다.

---

# 5. 패키지 충돌 검사


In [ ]:
!pip check


여기서 의존성 문제가 출력되면 내용을 확인하세요.

특히 다음과 같은 패키지에서 문제가 없어야 합니다.

```text
numpy
pandas
pyarrow
scipy
datasets
transformers
peft
trl
bitsandbytes
```

---

# 6. Hugging Face 로그인

MedGemma를 사용하기 전에 Hugging Face에서 MedGemma 접근 권한을 승인하고 Token을 준비해야 합니다.

Colab:

```text
왼쪽 메뉴
→ 열쇠 아이콘
→ Secrets
→ HF_TOKEN
```

으로 등록하세요.

코드에 Token을 직접 입력하지 않습니다.


In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN이 없습니다. "
        "Colab 왼쪽 메뉴의 Secrets에 HF_TOKEN을 등록하세요."
    )

login(token=HF_TOKEN)

print("Hugging Face login OK")


---

# 7. GitHub 저장소 Clone

## 현재는 실행하지 않습니다.

현재 MedGemma 학습 Notebook의 데이터셋과 학습 코드는 GitHub 프로젝트의 `ai/` 패키지를 사용하지 않습니다.

따라서 기존 Notebook의:

```python
!git clone ...
!pip install -e .
```

부분은 현재 단계에서는 불필요합니다.

나중에 `ai/consultation`, `ai/rag`, `ai/llm` 등의 공통 코드를 Colab에서 사용해야 할 때 추가하면
됩니다. (참고로 저장소는 `https://github.com/ThedaolOCR3/thegpt-project.git`, 브랜치는 `kbg`
— `main`엔 아직 `ai/ocr`, `ai/rag`가 없습니다.)

---

# 8. 데이터셋 로드

사용 데이터셋:

```text
sean0042/KorMedMCQA
```

직종별 config:

```text
doctor
nurse
pharm
dentist
```

현재는 의사 시험 데이터셋을 사용합니다.


In [ ]:
from datasets import load_dataset

DATASET_ID = "sean0042/KorMedMCQA"
DATASET_CONFIG = "doctor"

dataset = load_dataset(
    DATASET_ID,
    name=DATASET_CONFIG,
    split="train"
)

print(dataset)
print(dataset.column_names)


---

# 9. 데이터셋 샘플 확인


In [ ]:
print(dataset[0])
print("Dataset size:", len(dataset))


---

# 10. 데이터셋 컬럼 확인


In [ ]:
print(dataset.column_names)


예상되는 주요 컬럼:

```text
question
A
B
C
D
E
answer
```

`cot`이 존재하는 경우에는 해설도 사용할 수 있습니다.

---

# 11. 데이터셋 포맷팅

KorMedMCQA 데이터를 MedGemma의 대화 형식으로 변환합니다.


In [ ]:
OPTION_KEYS = ["A", "B", "C", "D", "E"]


def format_example(example):
    options_text = "\n".join(
        f"{key}. {example[key]}"
        for key in OPTION_KEYS
        if example.get(key) is not None
        and str(example[key]).strip() != ""
    )

    user_turn = (
        f"{example['question']}\n\n"
        f"선택지:\n"
        f"{options_text}"
    )

    answer_value = example["answer"]

    # answer가 문자열로 들어오는 경우도 대응
    if isinstance(answer_value, str):
        answer_value = answer_value.strip()

        if answer_value.upper() in OPTION_KEYS:
            answer_letter = answer_value.upper()
        else:
            answer_letter = OPTION_KEYS[int(answer_value) - 1]
    else:
        answer_letter = OPTION_KEYS[int(answer_value) - 1]

    answer_text = example.get(answer_letter, "")

    model_turn = (
        f"정답: {answer_letter}. {answer_text}"
    )

    cot = example.get("cot")

    if cot is not None and str(cot).strip():
        model_turn += f"\n\n해설: {cot}"

    return {
        "text": (
            "<start_of_turn>user\n"
            f"{user_turn}"
            "<end_of_turn>\n"
            "<start_of_turn>model\n"
            f"{model_turn}"
            "<end_of_turn>"
        )
    }


---

# 12. 데이터셋 변환


In [ ]:
dataset = dataset.map(
    format_example,
    remove_columns=[
        column
        for column in dataset.column_names
        if column != "text"
    ]
)


---

# 13. 변환 결과 확인


In [ ]:
print(dataset[0]["text"])


예상 형태:

```text
<start_of_turn>user
문제 내용

선택지:
A. ...
B. ...
C. ...
D. ...
E. ...
<end_of_turn>
<start_of_turn>model
정답: C. ...
<end_of_turn>
```

---

# 14. MedGemma 모델 설정


In [ ]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

MODEL_ID = "google/medgemma-4b-it"


---

# 15. 4bit 양자화 설정

## T4에서는 float16을 사용합니다.

T4(Turing 아키텍처)는 bf16 텐서코어 가속을 지원하지 않습니다(Ampere 이상부터 지원). 그래서
T4 환경에서 안정적/효율적으로 쓰려면 BF16 대신 FP16을 씁니다.


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


---

# 16. Tokenizer 로드


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
)

tokenizer.padding_side = "right"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


---

# 17. MedGemma 모델 로드


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,  # transformers 최신 버전은 torch_dtype 대신 dtype을 씀
    token=HF_TOKEN,
)


---

# 18. 모델 메모리 상태 확인


In [ ]:
print("Model loaded successfully")

if torch.cuda.is_available():
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

    print(
        "GPU memory reserved:",
        round(torch.cuda.memory_reserved() / 1024**3, 2),
        "GB"
    )


---

# 19. LoRA 설정


In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


---

# 20. K-bit Training 준비


In [ ]:
model = prepare_model_for_kbit_training(model)

model.config.use_cache = False


---

# 21. LoRA Configuration


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)


---

# 22. LoRA 적용


In [ ]:
model = get_peft_model(
    model,
    lora_config,
)

model.print_trainable_parameters()


출력 결과에서 전체 파라미터 대비 trainable parameter가 매우 적게 나오는 것이 정상입니다.

---

# 23. Google Drive 연결

Colab 세션이 종료되면 `/content`에 저장된 파일이 사라질 수 있습니다.

따라서 학습 체크포인트를 Google Drive에 저장합니다.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

CKPT_DIR = (
    "/content/drive/MyDrive/"
    "medgemma-lora-ckpt"
)

print("Checkpoint directory:", CKPT_DIR)


---

# 24. 학습 설정

## T4 16GB 기준


In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=CKPT_DIR,

    # T4 16GB
    per_device_train_batch_size=1,

    # 실제 batch size를 늘리는 효과
    gradient_accumulation_steps=8,

    # 처음 테스트할 때는 1 epoch 권장
    num_train_epochs=1,

    learning_rate=2e-4,

    # T4는 bf16 텐서코어가 없어서 bf16=False. fp16=True(AMP GradScaler)는 Gemma 계열에서
    # LoRA 레이어 일부가 bfloat16으로 생성되는 경우가 있어 GradScaler가 그 텐서를 처리 못 해
    # "_amp_foreach_non_finite_check_and_unscale_cuda not implemented for BFloat16" 에러가 남.
    # 4bit 베이스 자체가 이미 압축돼있고 LoRA 파라미터는 작아서, AMP 없이 기본 정밀도로 학습.
    fp16=False,
    bf16=False,

    # 메모리 절약
    gradient_checkpointing=True,

    logging_steps=10,

    # Colab 세션 종료 대비
    save_steps=50,
    save_total_limit=3,

    dataset_text_field="text",
    max_length=1024,

    # W&B 등의 외부 로깅 방지
    report_to="none",

    # 데이터 packing
    packing=False,
)


---

# 25. SFTTrainer 생성


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)


---

# 26. 학습 전 Trainer 확인


In [ ]:
print(trainer)


---

# 27. 학습 시작

## 처음에는 1 epoch로 테스트하는 것을 권장합니다.


In [ ]:
trainer.train()


학습이 정상적으로 시작되면 다음과 비슷한 로그가 출력됩니다.

```text
***** Running training *****
Num examples = ...
Num Epochs = 1
...
```

---

# 28. 학습 결과 저장

학습이 정상적으로 끝난 후 LoRA Adapter를 저장합니다.


In [ ]:
FINAL_ADAPTER_DIR = (
    "/content/drive/MyDrive/"
    "medgemma-lora-final"
)

trainer.save_model(FINAL_ADAPTER_DIR)

tokenizer.save_pretrained(
    FINAL_ADAPTER_DIR
)

print(
    "Adapter saved to:",
    FINAL_ADAPTER_DIR
)


---

# 29. 학습 체크포인트 확인


In [ ]:
import os

print(os.listdir(CKPT_DIR))


---

# 30. 세션이 끊긴 경우 체크포인트에서 재개

Colab 세션이 종료된 경우:

1. 런타임 재연결
2. 패키지 설치
3. 런타임 재시작
4. HF 로그인
5. 데이터셋 로드
6. 모델 로드
7. LoRA 설정
8. Drive mount
9. Trainer 생성

까지 다시 실행합니다.

그 다음:


In [ ]:
import os

checkpoint_dirs = [
    os.path.join(CKPT_DIR, name)
    for name in os.listdir(CKPT_DIR)
    if name.startswith("checkpoint-")
]

checkpoint_dirs = [
    path
    for path in checkpoint_dirs
    if os.path.isdir(path)
]

if checkpoint_dirs:
    checkpoint_dirs.sort(
        key=lambda path: int(
            os.path.basename(path).split("-")[-1]
        )
    )

    latest_checkpoint = checkpoint_dirs[-1]

    print(
        "Latest checkpoint:",
        latest_checkpoint
    )

    trainer.train(
        resume_from_checkpoint=latest_checkpoint
    )

else:
    print("체크포인트가 없습니다. 처음부터 학습을 시작하세요.")


---

# 31. 간단한 추론 테스트

학습이 완료되었으면 먼저 Colab 안에서 모델이 제대로 답변하는지 테스트합니다.


추론 전에 학습 때 꺼뒀던 캐시를 다시 켭니다 — 안 켜면 두통이 3일째 있어요 같은
질문에 같은 글자만 반복하는("두두두두...") 증상이 날 수 있습니다.


In [ ]:
model.eval()
model.config.use_cache = True
model.gradient_checkpointing_disable()


In [ ]:
prompt = (
    "<start_of_turn>user\n"
    "두통이 3일째 있어요. "
    "어떤 진료과를 방문하는 것이 좋을까요?"
    "<end_of_turn>\n"
    "<start_of_turn>model\n"
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
    )

result = tokenizer.decode(
    output[0],
    skip_special_tokens=True,
)

print(result)


---

# 32. 다른 질문으로 테스트


In [ ]:
test_questions = [
    "3일째 열이 나고 기침이 있어요.",
    "가슴이 갑자기 아프고 숨쉬기가 힘들어요.",
    "복통이 계속되는데 어느 진료과에 가야 하나요?",
]

for question in test_questions:

    prompt = (
        "<start_of_turn>user\n"
        f"{question}"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )

    result = tokenizer.decode(
        output[0],
        skip_special_tokens=True,
    )

    print("=" * 80)
    print("질문:", question)
    print(result)


---

# 33. Hugging Face Hub에 LoRA Adapter 업로드

## 먼저 Hugging Face에서 업로드할 Repository를 준비하세요.

예:

```text
your-hf-account/medgemma-4b-lora-consultation
```

아래의 `<HF계정>`을 실제 계정명으로 변경합니다.


In [ ]:
ADAPTER_REPO = (
    "<HF계정>/medgemma-4b-lora-consultation"
)


---

# 34. Adapter 업로드


In [ ]:
model.push_to_hub(
    ADAPTER_REPO,
    private=True,
)

tokenizer.push_to_hub(
    ADAPTER_REPO,
    private=True,
)

print(
    "Uploaded to Hugging Face:",
    ADAPTER_REPO
)


---

# 35. 중요: GitHub와 Hugging Face의 역할

이 Notebook에서 GitHub는 학습 코드 관리용입니다.

```text
GitHub
├── train_medgemma_lora.ipynb
├── training code
└── configuration
```

학습된 모델 Adapter는 Hugging Face에 저장합니다.

```text
Hugging Face
└── medgemma-4b-lora-consultation
    ├── adapter_config.json
    ├── adapter_model.safetensors
    └── tokenizer files
```

GitHub에 대용량 모델 파일을 직접 올리지 않습니다.

---

# 36. 이후 FastAPI에서 사용하는 구조

학습이 끝난 후에는:

```text
MedGemma Base Model
        +
LoRA Adapter
        ↓
Fine-tuned MedGemma
        ↓
FastAPI
        ↓
POST /api/v1/ai/chat
```

구조로 사용할 수 있습니다.

**아래는 코드가 아니라 설명용 스니펫입니다 — 이 셀은 실행하지 마세요.** `base_model = ...`은
자리표시일 뿐 실제로 동작하는 코드가 아니고, 이건 이 학습 노트북이 아니라 나중에
`ai/llm`(현재 빈 패키지)에 서버 쪽 로딩 코드를 짤 때 참고할 형태입니다:

```python
from peft import PeftModel

base_model = ...  # 서버 쪽에서 medgemma-4b-it을 로드한 것

model = PeftModel.from_pretrained(
    base_model,
    "<HF계정>/medgemma-4b-lora-consultation"
)
```


---

# 37. 학습 데이터에 대한 주의사항

현재 사용하는:

```text
sean0042/KorMedMCQA
```

데이터셋은 실험 단계에서 사용할 수 있지만, **라이선스가 CC-BY-NC-2.0(비영리 조건)** 입니다.

실제 상용 서비스로 발전시키는 경우 데이터셋 라이선스와 모델 라이선스를 별도로 검토해야 합니다.

또한 실제 환자 개인정보가 포함된 데이터는 GitHub나 공개 Hugging Face Repository에 업로드하지
마세요.

---

# 38. 최종 전체 흐름

```text
Google Colab
    │
    ├── T4 GPU
    │
    ├── NumPy (Colab 기본값, 강제로 안 낮춤)
    │
    ├── KorMedMCQA
    │
    ├── MedGemma 4B
    │
    ├── 4bit Quantization
    │
    └── LoRA
            │
            ▼
       Fine-tuning
            │
            ▼
       Google Drive
       Checkpoint
            │
            ▼
       LoRA Adapter
            │
            ▼
      Hugging Face Hub
            │
            ▼
       FastAPI / Cloud Run
            │
            ▼
       실제 AI 서비스
```

# 핵심 체크리스트

* [ ] Colab GPU가 T4로 연결되어 있는가?
* [ ] numpy 버전을 억지로 낮추지 않고 Colab 기본값을 그대로 쓰고 있는가?
* [ ] 패키지 설치 후 런타임을 재시작했는가?
* [ ] `from datasets import load_dataset`가 정상 실행되는가?
* [ ] `pip check`에서 심각한 dependency conflict가 없는가?
* [ ] Hugging Face `HF_TOKEN`이 Colab Secrets에 등록되어 있는가?
* [ ] MedGemma 접근 권한이 있는가?
* [ ] GitHub clone은 현재 단계에서 생략했는가?
* [ ] Google Drive가 연결되어 있는가?
* [ ] T4에서 `fp16=False`, `bf16=False`로 설정했는가? (fp16 AMP는 Gemma 계열과 GradScaler 충돌 있음)
* [ ] 학습 전 Base Model 평가를 준비했는가?
* [ ] 학습 후 LoRA Adapter를 저장했는가?
* [ ] Hugging Face Repository를 private으로 설정했는가?
* [ ] 실제 서비스에서는 FastAPI가 Adapter를 불러오도록 구성했는가?


---

# 39. (선택) 재학습 없이 어댑터만 불러와서 빠르게 테스트

**세션이 끊겨서 다시 들어왔거나, 학습은 이미 끝나서 어댑터가 HF Hub에 올라가 있는 상태라면
이 섹션부터 실행하면 됩니다.** 데이터셋 로드/포맷팅(4~5번)이나 학습(9번) 전체를 다시 안 해도
됩니다 — 아래 5개 셀만 순서대로 실행하면 곧바로 테스트할 수 있습니다.

1. 패키지 설치(1번) → **런타임 다시 시작** → HF 로그인(2번)까지는 그대로 필요
2. 그다음 아래 셀들만 실행:


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "google/medgemma-4b-it"
ADAPTER_REPO = "gon-0130/medgemma-4b-lora-consultation"  # 실제 올린 이름으로 확인

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    token=HF_TOKEN,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.eval()
model.config.use_cache = True

print("어댑터 로드 완료")


## 39-1. 반복 증상("두두두...", "정정정...") 잡기 위한 디코딩 옵션

`use_cache`를 켜도 같은 증상이면, 캐시 문제가 아니라 **탐욕적(greedy) 디코딩이 학습으로 좁아진
확률분포에 갇혀서** 그럴 가능성이 큽니다(1 epoch·짧고 반복적인 학습 타깃 특성상 흔함).
`repetition_penalty`와 `no_repeat_ngram_size`로 같은 토큰 반복을 명시적으로 막고,
`do_sample=True`로 약간의 무작위성을 줘서 탈출 가능성을 높입니다.


In [ ]:
def ask(question):
    prompt = (
        "<start_of_turn>user\n"
        f"{question}"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


print(ask("두통이 3일째 있어요. 어떤 진료과를 방문하는 것이 좋을까요?"))


**이래도 여전히 반복되면** 디코딩 설정 문제가 아니라 학습 자체가 부족했을 가능성이 큽니다
(1 epoch, 학습 데이터의 답변이 대부분 짧고 정형화돼 있었음 — "11. 데이터셋 포맷팅" 셀 설명
참고). 그땐 `num_train_epochs`를 2~3으로 늘리거나, `cot`(해설)가 있는 데이터 비중을 늘려서
다시 학습하는 걸 고려해야 합니다.
